### 📈 Why Trading Strategies Don't Need to be Secretive

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 📈 *Example:* Beating the Market with a Trading Strategy

Allocating risk is literally stepping up to the plate, it is no different, some are better than others at preforming

The first question to you need to answer is *what* are you trading?

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 40
np.random.seed(SEED)

N_YEARS = 2      # only 2 years forward (was 5)
TRADING_DAYS_PER_YEAR = 252
N_DAYS = N_YEARS * TRADING_DAYS_PER_YEAR
DT = 1 / TRADING_DAYS_PER_YEAR

INITIAL_VALUE = 100
START_DATE = "2026-01-01"
RISK_FREE_RATE = 0.00

# ------------------------------------------------------------
# Market regime flag
# ------------------------------------------------------------
MARKET_REGIME = "bear"

# In bear regime, strategy has large negative alpha and higher beta, simulating overexposure to downside
REGIME_CONFIGS = {
    "bull": {
        "market_mu": 0.11,
        "market_sigma": 0.17,
        "strategy_alpha": 0.015,
        "strategy_beta": 1.35,
        "description": "",
    },
    "bear": {
        "market_mu": -0.10,
        "market_sigma": 0.25,
        "strategy_alpha": -0.06,      # significantly negative idiosyncratic alpha
        "strategy_beta": 2.3,         # huge overexposure to market beta
        "description": "",
    },
    "sideways": {
        "market_mu": 0.00,
        "market_sigma": 0.16,
        "strategy_alpha": -0.010,
        "strategy_beta": 1.35,
        "description": "",
    },
}

if MARKET_REGIME.lower() not in REGIME_CONFIGS:
    raise ValueError("MARKET_REGIME must be one of: 'bull', 'bear', or 'sideways'.")

REGIME = REGIME_CONFIGS[MARKET_REGIME.lower()]
MARKET_MU = REGIME["market_mu"]
MARKET_SIGMA = REGIME["market_sigma"]
STRATEGY_ALPHA = REGIME["strategy_alpha"]
STRATEGY_BETA = REGIME["strategy_beta"]

STRATEGY_IDIOSYNCRATIC_VOL = 0.10

FRAME_STRIDE = 5
FRAME_DURATION = 20
MIN_REGRESSION_OBS = 20

OUTPUT_HTML = "market_strategy_beta_regime_animation.html"
SHOW_FIG = True

# ============================================================
# Simulate market plus beta-dependent strategy return stream
# ============================================================

dates = pd.bdate_range(start=START_DATE, periods=N_DAYS + 1)

z_market = np.random.normal(size=N_DAYS)
z_idio = np.random.normal(size=N_DAYS)

market_log_return = (
    (MARKET_MU - 0.5 * MARKET_SIGMA**2) * DT
    + MARKET_SIGMA * np.sqrt(DT) * z_market
)
market_return = np.exp(market_log_return) - 1

strategy_return = (
    STRATEGY_ALPHA * DT
    + STRATEGY_BETA * market_return
    + STRATEGY_IDIOSYNCRATIC_VOL * np.sqrt(DT) * z_idio
)

strategy_return = np.clip(strategy_return, -0.95, None)

market = INITIAL_VALUE * np.r_[1, np.cumprod(1 + market_return)]
strategy = INITIAL_VALUE * np.r_[1, np.cumprod(1 + strategy_return)]

df = pd.DataFrame({
    "date": dates,
    "Market": market,
    "Trading Strategy": strategy,
})

df["Market Return"] = df["Market"].pct_change()
df["Trading Strategy Return"] = df["Trading Strategy"].pct_change()

df["Market Drawdown"] = df["Market"] / df["Market"].cummax() - 1
df["Trading Strategy Drawdown"] = df["Trading Strategy"] / df["Trading Strategy"].cummax() - 1

df["Relative Outperformance"] = df["Trading Strategy"] / df["Market"] - 1
returns_df = df.dropna().reset_index(drop=True)
latest_date = df["date"].iloc[-1]

# ============================================================
# Helpers
# ============================================================

def annualized_sharpe(returns, risk_free_rate=0.0):
    daily_rf = (1 + risk_free_rate) ** (1 / TRADING_DAYS_PER_YEAR) - 1
    excess = returns.dropna() - daily_rf
    if len(excess) < 2 or np.isclose(excess.std(ddof=0), 0):
        return 0.0
    return excess.mean() / excess.std(ddof=0) * np.sqrt(TRADING_DAYS_PER_YEAR)

def max_drawdown(series):
    return (series / series.cummax() - 1).min()

def performance_metrics(price_series, return_series):
    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    cagr = (price_series.iloc[-1] / price_series.iloc[0]) ** (1 / N_YEARS) - 1
    ann_vol = return_series.dropna().std(ddof=0) * np.sqrt(TRADING_DAYS_PER_YEAR)
    sharpe = annualized_sharpe(return_series, RISK_FREE_RATE)
    mdd = max_drawdown(price_series)
    # Return the MDD as positive number for the bar chart!
    return np.array([
        total_return * 100,
        cagr * 100,
        ann_vol * 100,
        abs(mdd) * 100,    # Make MDD a positive value
        sharpe,
    ])

def format_metric_text(values):
    return [
        f"{values[0]:.1f}%",
        f"{values[1]:.1f}%",
        f"{values[2]:.1f}%",
        f"{values[3]:.1f}%",
        f"{values[4]:.2f}",
    ]

def padded_range(values, pad_fraction=0.08, min_pad=1.0):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]

def drawdown_range(drawdown_series):
    dd_min = float(drawdown_series.min())
    lower = min(dd_min * 1.15, -0.02)
    return [lower, 0.02]

def regression_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.isclose(np.std(x), 0):
        intercept = float(np.mean(y)) if len(y) else 0.0
        return 0.0, intercept, 0.0, 0.0
    beta, intercept = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1]
    r2 = corr**2
    return float(beta), float(intercept), float(corr), float(r2)

def annualize_daily_alpha(daily_alpha):
    if daily_alpha <= -1:
        return -1.0
    return (1 + daily_alpha) ** TRADING_DAYS_PER_YEAR - 1

def regression_text(beta, intercept, corr, r2):
    ann_alpha = annualize_daily_alpha(intercept)
    return (
        f"β = {beta:.2f}<br>"
        f"Corr = {corr:.2f} · R² = {r2:.2f}<br>"
        f"Ann. α = {ann_alpha:.1%}"
    )

# ============================================================
# Initial/Final Summary metrics and axis ranges
# ============================================================

metric_labels = ["Total Return", "CAGR", "Ann Vol", "Max DD", "Sharpe"]

market_metrics = performance_metrics(df["Market"], df["Market Return"])
strategy_metrics = performance_metrics(df["Trading Strategy"], df["Trading Strategy Return"])

final_beta, final_intercept, realized_corr, final_r2 = regression_stats(
    returns_df["Market Return"],
    returns_df["Trading Strategy Return"],
)
final_ann_alpha = annualize_daily_alpha(final_intercept)
final_relative_outperformance = df["Relative Outperformance"].iloc[-1]

price_range = padded_range(np.r_[df["Market"].values, df["Trading Strategy"].values], 0.08)
market_dd_range = drawdown_range(df["Market Drawdown"])
strategy_dd_range = drawdown_range(df["Trading Strategy Drawdown"])

# Dynamically set metric_range based on terminal (final) metric values
terminal_metrics = np.r_[market_metrics[:4], strategy_metrics[:4]]
terminal_max = np.nanmax(terminal_metrics)
terminal_min = np.nanmin(terminal_metrics)
pad = (terminal_max - terminal_min) * 0.18 if not np.isclose(terminal_max, terminal_min) else max(abs(terminal_max) * 0.18, 1.0)
metric_range = [terminal_min - pad, terminal_max + pad]

x_min = float(returns_df["Market Return"].min())
x_max = float(returns_df["Market Return"].max())
x_pad = max((x_max - x_min) * 0.15, 0.005)
scatter_x_range = [x_min - x_pad, x_max + x_pad]

y_min = float(returns_df["Trading Strategy Return"].min())
y_max = float(returns_df["Trading Strategy Return"].max())
y_pad = max((y_max - y_min) * 0.15, 0.005)
scatter_y_range = [y_min - y_pad, y_max + y_pad]

reg_x = np.array(scatter_x_range)

text_x = scatter_x_range[0] + 0.05 * (scatter_x_range[1] - scatter_x_range[0])
text_y = scatter_y_range[1] - 0.10 * (scatter_y_range[1] - scatter_y_range[0])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
market_color = "#00d4ff"
strategy_color = "#00ff88"
scatter_color = "rgba(224,224,224,0.68)"
regression_color = "#ffaa33"
baseline_color = "#777777"
drawdown_line_red = "#ff2222"
drawdown_fill_red = "rgba(255,60,60,0.24)"  # More visible red fill for drawdown

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=3,
    cols=2,
    # Increased this vertical_spacing to add more padding beneath first row of charts
    row_heights=[0.46, 0.32, 0.22],  # More space for drawdowns (middle row)
    column_widths=[0.52, 0.48],
    horizontal_spacing=0.08,
    vertical_spacing=0.16,   # <--- was 0.09, now 0.16 for more space below row 1
    subplot_titles=("", "", "", "", "", ""),  # Remove subtitles
    specs=[
        [{}, {}],
        [{}, {}],
        [{"secondary_y": True}, {"secondary_y": True}],  # For twin y on bottom row for bar+Sharpe
    ],
)

initial_i = max(2, MIN_REGRESSION_OBS)
initial_slice = df.iloc[: initial_i + 1]
initial_returns = returns_df.iloc[:initial_i]

initial_beta, initial_intercept, initial_corr, initial_r2 = regression_stats(
    initial_returns["Market Return"],
    initial_returns["Trading Strategy Return"],
)
initial_reg_y = initial_intercept + initial_beta * reg_x

# Top-left: cumulative performance
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Market"],
        mode="lines",
        line=dict(color=market_color, width=3),
        name="Market",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Market: %{y:.2f}<extra></extra>",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Trading Strategy"],
        mode="lines",
        line=dict(color=strategy_color, width=3),
        name="Trading Strategy",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Trading Strategy: %{y:.2f}<extra></extra>",
    ),
    row=1, col=1,
)

# Top-right: animated scatter
fig.add_trace(
    go.Scatter(
        x=initial_returns["Market Return"],
        y=initial_returns["Trading Strategy Return"],
        mode="markers",
        marker=dict(size=6, color=scatter_color, line=dict(width=0)),
        customdata=initial_returns["date"],
        name="Return Scatter",
        showlegend=False,
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Market Return: %{x:.2%}<br>"
            "Trading Strategy Return: %{y:.2%}<extra></extra>"
        ),
    ),
    row=1, col=2,
)

# Expanding OLS regression line.
fig.add_trace(
    go.Scatter(
        x=reg_x,
        y=initial_reg_y,
        mode="lines",
        line=dict(color=regression_color, width=3, dash="dash"),
        name="Expanding OLS Fit",
        showlegend=False,
        hovertemplate="Expanding OLS fit<extra></extra>",
    ),
    row=1, col=2,
)
# Removed regression stats text overlay on regression plot

# Middle row: drawdowns (now both red lines with red shading)
for drawdown_col, title in [("Market Drawdown", "Market Drawdown"), ("Trading Strategy Drawdown", "Trading Strategy Drawdown")]:
    fig.add_trace(
        go.Scatter(
            x=initial_slice["date"],
            y=initial_slice[drawdown_col],
            mode="lines",
            line=dict(color=drawdown_line_red, width=2),
            fill="tozeroy",
            fillcolor=drawdown_fill_red,
            showlegend=False,
            hovertemplate=f"Date: "+"%{{x|%Y-%m-%d}}<br>{title}: "+"%{{y:.2%}}<extra></extra>",
        ),
        row=2,
        col=1 if drawdown_col == "Market Drawdown" else 2,
    )

# Bottom row: animated bar charts (metrics)
# We'll plot the first metrics, but will create frames for animation

def bar_metrics(price_series, return_series):
    vals = performance_metrics(price_series, return_series)
    percent_vals = vals[:4]
    sharpe_val = vals[4]
    return percent_vals, sharpe_val, format_metric_text(vals)

mkt_barvals, mkt_sharpeval, mkt_text = bar_metrics(df["Market"].iloc[:initial_i+1], df["Market Return"].iloc[:initial_i+1])
strat_barvals, strat_sharpeval, strat_text = bar_metrics(df["Trading Strategy"].iloc[:initial_i+1], df["Trading Strategy Return"].iloc[:initial_i+1])

# Market bar: primary y for 4 percent metrics, secondary y for Sharpe bar
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=mkt_barvals,
        marker=dict(color=[market_color]*4, opacity=0.84),
        showlegend=False,
        text=mkt_text[:4],
        textposition="auto",
        name="Market Metrics",
        customdata=mkt_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=1, secondary_y=False,
)
fig.add_trace(
    go.Bar(
        x=[metric_labels[4]],
        y=[mkt_sharpeval],
        marker=dict(color=["#e865e8"], opacity=0.85),
        showlegend=False,
        text=[mkt_text[4]],
        textposition="auto",
        name="Market Sharpe",
        customdata=[mkt_text[4]],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=1, secondary_y=True,
)

# Strategy bar: primary y for 4 percent metrics, secondary y for Sharpe bar
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=strat_barvals,
        marker=dict(color=[strategy_color]*4, opacity=0.84),
        showlegend=False,
        text=strat_text[:4],
        textposition="auto",
        name="Strategy Metrics",
        customdata=strat_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=2, secondary_y=False,
)
fig.add_trace(
    go.Bar(
        x=[metric_labels[4]],
        y=[strat_sharpeval],
        marker=dict(color=["#e865e8"], opacity=0.85),
        showlegend=False,
        text=[strat_text[4]],
        textposition="auto",
        name="Strategy Sharpe",
        customdata=[strat_text[4]],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=2, secondary_y=True,
)

# Reference lines
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_vline(x=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=1)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=1, secondary_y=False)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=2, secondary_y=False)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(initial_i, len(df), FRAME_STRIDE))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

# Only animate up to 2 years (~504 trading days)
max_frame = TRADING_DAYS_PER_YEAR * N_YEARS  # 504 if 2 years
frame_indices = [i for i in frame_indices if i <= max_frame]
if len(frame_indices) == 0 or frame_indices[-1] != max_frame:
    frame_indices.append(max_frame)
frame_indices = [i for i in frame_indices if i >= initial_i and i <= max_frame]

for i in frame_indices:
    frame_name = f"f{i}"
    current_slice = df.iloc[: i + 1]
    current_returns = returns_df.iloc[:i]

    # Metrics (bar) per frame, for current slice
    mkt_barvals, mkt_sharpeval, mkt_text = bar_metrics(current_slice["Market"], current_slice["Market Return"])
    strat_barvals, strat_sharpeval, strat_text = bar_metrics(current_slice["Trading Strategy"], current_slice["Trading Strategy Return"])

    beta, intercept, corr, r2 = regression_stats(
        current_returns["Market Return"],
        current_returns["Trading Strategy Return"],
    )
    reg_y = intercept + beta * reg_x

    frames.append(
        go.Frame(
            data=[
                go.Scatter(x=current_slice["date"], y=current_slice["Market"]),   # 0
                go.Scatter(x=current_slice["date"], y=current_slice["Trading Strategy"]),  # 1
                go.Scatter(
                    x=current_returns["Market Return"],
                    y=current_returns["Trading Strategy Return"],
                    customdata=current_returns["date"],
                ),                                                               # 2
                go.Scatter(x=reg_x, y=reg_y),                                    # 3
                # Removed regression stats text overlay in animation frames
                # Drawdowns (both red with fill)
                go.Scatter(
                    x=current_slice["date"],
                    y=current_slice["Market Drawdown"],
                    line=dict(color=drawdown_line_red, width=2),
                    fill="tozeroy",
                    fillcolor=drawdown_fill_red,
                ),                                                               # 4
                go.Scatter(
                    x=current_slice["date"],
                    y=current_slice["Trading Strategy Drawdown"],
                    line=dict(color=drawdown_line_red, width=2),
                    fill="tozeroy",
                    fillcolor=drawdown_fill_red,
                ),                                                               # 5
                # Metrics percent bars (primary y) and Sharpe bars (secondary y) for Market and Strategy
                go.Bar(
                    x=metric_labels[:4],
                    y=mkt_barvals,
                    marker=dict(color=[market_color]*4, opacity=0.84),
                    text=mkt_text[:4],
                    textposition="auto",
                    customdata=mkt_text[:4],
                ),                                                               # 6 (market, y)
                go.Bar(
                    x=[metric_labels[4]],
                    y=[mkt_sharpeval],
                    marker=dict(color=["#e865e8"], opacity=0.85),
                    text=[mkt_text[4]],
                    textposition="auto",
                    customdata=[mkt_text[4]],
                ),                                                               # 7 (market, secondary_y)
                go.Bar(
                    x=metric_labels[:4],
                    y=strat_barvals,
                    marker=dict(color=[strategy_color]*4, opacity=0.84),
                    text=strat_text[:4],
                    textposition="auto",
                    customdata=strat_text[:4],
                ),                                                               # 8 (strat, y)
                go.Bar(
                    x=[metric_labels[4]],
                    y=[strat_sharpeval],
                    marker=dict(color=["#e865e8"], opacity=0.85),
                    text=[strat_text[4]],
                    textposition="auto",
                    customdata=[strat_text[4]],
                ),                                                               # 9 (strat, secondary_y)
            ],
            traces=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
            name=frame_name,
        )
    )

    elapsed_years = i / TRADING_DAYS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="High-Beta Trading Strategy Reliance on the Market Return Stream",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=145, b=150, r=50, l=75),
    legend=dict(
        orientation="v",
        x=0,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right",
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=13))

# Cumulative performance
fig.update_xaxes(axis_style, row=1, col=1, range=[df["date"].iloc[0], df["date"].iloc[-1]], title_text="Date")
fig.update_yaxes(axis_style, row=1, col=1, range=price_range, title_text="Index Value")

# Scatter axes
fig.update_xaxes(axis_style, row=1, col=2, range=scatter_x_range, title_text="Market Daily Return", tickformat=".1%")
fig.update_yaxes(axis_style, row=1, col=2, range=scatter_y_range, title_text="Trading Strategy Daily Return", tickformat=".1%")

# Drawdown
fig.update_xaxes(axis_style, row=2, col=1, range=[df["date"].iloc[0], df["date"].iloc[-1]], title_text="")
fig.update_yaxes(axis_style, row=2, col=1, range=market_dd_range, title_text="Drawdown", tickformat=".0%", color="#ff2222", linecolor="#ff2222", tickfont=dict(color="#ffbbbb"))

fig.update_xaxes(axis_style, row=2, col=2, range=[df["date"].iloc[0], df["date"].iloc[-1]], title_text="")
fig.update_yaxes(axis_style, row=2, col=2, range=strategy_dd_range, title_text="Drawdown", tickformat=".0%", color="#ff2222", linecolor="#ff2222", tickfont=dict(color="#ffbbbb"))

# Performance metric bar (primary y: percent; secondary y: Sharpe)
fig.update_xaxes(axis_style, row=3, col=1, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3, col=1,
    range=metric_range,
    title_text="Percent values",
    secondary_y=False,
)
fig.update_yaxes(
    dict(
        title_text="",  # Remove label for twiny (Sharpe) axis on market bar chart
        showgrid=False,
        showline=True,
        zeroline=False,
        tickfont=dict(color="#e865e8"),
        title_font=dict(color="#e865e8"),
        rangemode="tozero",
        range=[0, 1.5],  # <--- Fixed y-lim for Sharpe twin axis
    ),
    row=3, col=1, secondary_y=True,
)

fig.update_xaxes(axis_style, row=3, col=2, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3, col=2,
    range=metric_range,
    title_text="Percent values",
    secondary_y=False,
)
fig.update_yaxes(
    dict(
        title_text="Sharpe Ratio",
        showgrid=False,
        showline=True,
        zeroline=False,
        tickfont=dict(color="#e865e8"),
        title_font=dict(color="#e865e8"),
        rangemode="tozero",
        range=[0, 1.5],  # <--- Fixed y-lim for Sharpe twin axis
    ),
    row=3, col=2, secondary_y=True,
)

# ============================================================
# Save / show
# ============================================================

fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn")

if SHOW_FIG:
    fig.show()


###### ______________________________________________________________________________________________________________________________________

 

##### 🧢 Undiversifiable Risk is a Pitch

Returns can be explained in the cross-section by factor models attempting to discern priced risk.  *These models **fail** to predict returns*.

 $$ r_{it} = \alpha_i + \sum_{k=1}^K \beta_{ik} f_{kt} + \epsilon_{it} $$


The literature documents many risk premia including...
 
 - **MRP** (Market Risk Premium): The excess return investors demand for taking exposure to the broad equity market versus a risk-free asset.
 - **VRP** (Variance Risk Premium): Compensation for bearing volatility risk, typically captured by strategies selling options or variance swaps.
 - **SMB** (Small-Minus-Big): The size premium, representing returns to small-cap stocks over large-cap stocks (Fama-French).
 - **HML** (High-Minus-Low): The value premium, for holding high book-to-market (value) stocks over low (growth) stocks (Fama-French).
 - **MOM** (Momentum): The tendency of assets with strong recent returns to continue outperforming in the near future (Carhart 4-factor model).
 - **L/S Equity Factors:** Profitability, investment, and other style factors often studied in the literature.
 - **Credit Risk Premium:** The excess yield of corporate bonds over risk-free Treasuries (compensation for default risk).
 - **Term Premium:** The excess return from holding long-term bonds instead of rolling over short-term bonds.
 - **FX Carry and Risk Premia:** Returns earned by borrowing in low-yielding currencies and investing in high-yielding ones.
 - **Liquidity Premium:** Compensation for holding less liquid assets.
 
 These risk premia represent broad, systematic return sources, not idiosyncratic (stock-specific) alpha.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 42
np.random.seed(SEED)

N_YEARS = 2
TRADING_DAYS_PER_YEAR = 252
N_DAYS = N_YEARS * TRADING_DAYS_PER_YEAR
DT = 1 / TRADING_DAYS_PER_YEAR

INITIAL_VALUE = 100
START_DATE = "2026-01-01"
RISK_FREE_RATE = 0.00

# ------------------------------------------------------------
# Market regime flag
# ------------------------------------------------------------
MARKET_REGIME = "bear"

REGIME_CONFIGS = {
    "bull": {
        "market_mu": 0.11,
        "market_sigma": 0.17,
        "description": "Bull market: positive market drift, sentiment remains independent",
    },
    "bear": {
        "market_mu": -0.10,
        "market_sigma": 0.25,
        "description": "Bear market: negative market drift, sentiment remains independent",
    },
    "sideways": {
        "market_mu": 0.00,
        "market_sigma": 0.16,
        "description": "Sideways market: limited market drift, sentiment remains independent",
    },
}

if MARKET_REGIME.lower() not in REGIME_CONFIGS:
    raise ValueError("MARKET_REGIME must be one of: 'bull', 'bear', or 'sideways'.")

REGIME = REGIME_CONFIGS[MARKET_REGIME.lower()]
MARKET_MU = REGIME["market_mu"]
MARKET_SIGMA = REGIME["market_sigma"]

# Sentiment premium is deliberately independent of the market regime.
SENTIMENT_MU = 0.055
SENTIMENT_SIGMA = 0.20
SENTIMENT_MARKET_BETA = 0.00
ORTHOGONALIZE_SHOCKS = True

FRAME_STRIDE = 5
FRAME_DURATION = 20
MIN_REGRESSION_OBS = 20

OUTPUT_HTML = "orthogonal_market_sentiment_regime_animation.html"
SHOW_FIG = False

# ============================================================
# Simulate two orthogonal risk premia
# ============================================================

dates = pd.bdate_range(start=START_DATE, periods=N_DAYS + 1)

z_market = np.random.normal(size=N_DAYS)
z_sentiment_raw = np.random.normal(size=N_DAYS)

if ORTHOGONALIZE_SHOCKS:
    z_market_centered = z_market - z_market.mean()
    z_sentiment_centered = z_sentiment_raw - z_sentiment_raw.mean()
    projection = (
        np.dot(z_sentiment_centered, z_market_centered)
        / np.dot(z_market_centered, z_market_centered)
    ) * z_market_centered
    z_sentiment = z_sentiment_centered - projection
    z_sentiment = z_sentiment / z_sentiment.std(ddof=0)
else:
    z_sentiment = z_sentiment_raw

market_log_return = (
    (MARKET_MU - 0.5 * MARKET_SIGMA**2) * DT
    + MARKET_SIGMA * np.sqrt(DT) * z_market
)
market_return = np.exp(market_log_return) - 1

sentiment_log_return = (
    (SENTIMENT_MU - 0.5 * SENTIMENT_SIGMA**2) * DT
    + SENTIMENT_SIGMA * np.sqrt(DT) * z_sentiment
    + SENTIMENT_MARKET_BETA * market_return
)
sentiment_return = np.exp(sentiment_log_return) - 1
sentiment_return = np.clip(sentiment_return, -0.95, None)

market = INITIAL_VALUE * np.r_[1, np.cumprod(1 + market_return)]
sentiment = INITIAL_VALUE * np.r_[1, np.cumprod(1 + sentiment_return)]

df = pd.DataFrame({
    "date": dates,
    "Market": market,
    "Sentiment": sentiment,
})

df["Market Return"] = df["Market"].pct_change()
df["Sentiment Return"] = df["Sentiment"].pct_change()

df["Market Drawdown"] = df["Market"] / df["Market"].cummax() - 1
df["Sentiment Drawdown"] = df["Sentiment"] / df["Sentiment"].cummax() - 1

df["Relative Sentiment vs Market"] = df["Sentiment"] / df["Market"] - 1
returns_df = df.dropna().reset_index(drop=True)
latest_date = df["date"].iloc[-1]

# ============================================================
# Helpers
# ============================================================

def annualized_sharpe(returns, risk_free_rate=0.0):
    daily_rf = (1 + risk_free_rate) ** (1 / TRADING_DAYS_PER_YEAR) - 1
    excess = returns.dropna() - daily_rf
    if len(excess) < 2 or np.isclose(excess.std(ddof=0), 0):
        return 0.0
    return excess.mean() / excess.std(ddof=0) * np.sqrt(TRADING_DAYS_PER_YEAR)

def max_drawdown(series):
    return (series / series.cummax() - 1).min()

def performance_metrics(price_series, return_series, years_elapsed=None):
    price_series = price_series.dropna()
    return_series = return_series.dropna()
    if years_elapsed is None:
        years_elapsed = max(len(return_series) / TRADING_DAYS_PER_YEAR, 1 / TRADING_DAYS_PER_YEAR)
    else:
        years_elapsed = max(years_elapsed, 1 / TRADING_DAYS_PER_YEAR)
    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    cagr = (price_series.iloc[-1] / price_series.iloc[0]) ** (1 / years_elapsed) - 1
    ann_vol = return_series.std(ddof=0) * np.sqrt(TRADING_DAYS_PER_YEAR) if len(return_series) > 1 else 0.0
    sharpe = annualized_sharpe(return_series, RISK_FREE_RATE)
    mdd = max_drawdown(price_series)
    return np.array([
        total_return * 100,
        cagr * 100,
        ann_vol * 100,
        abs(mdd) * 100,
        sharpe,
    ])

def format_metric_text(values):
    return [
        f"{values[0]:.1f}%",
        f"{values[1]:.1f}%",
        f"{values[2]:.1f}%",
        f"{values[3]:.1f}%",
        f"{values[4]:.2f}",
    ]

def padded_range(values, pad_fraction=0.08, min_pad=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]

def drawdown_range(drawdown_series):
    dd_min = float(drawdown_series.min())
    lower = min(dd_min * 1.15, -0.02)
    return [lower, 0.02]

def regression_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.isclose(np.std(x), 0):
        intercept = float(np.mean(y)) if len(y) else 0.0
        return 0.0, intercept, 0.0, 0.0
    beta, intercept = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1]
    r2 = corr**2
    return float(beta), float(intercept), float(corr), float(r2)

def annualize_daily_intercept(daily_intercept):
    if daily_intercept <= -1:
        return -1.0
    return (1 + daily_intercept) ** TRADING_DAYS_PER_YEAR - 1

def regression_text(beta, intercept, corr, r2):
    ann_intercept = annualize_daily_intercept(intercept)
    return (
        f"β = {beta:.2f}<br>"
        f"Corr = {corr:.2f} · R² = {r2:.2f}<br>"
        f"Ann. intercept = {ann_intercept:.1%}"
    )

def elapsed_years_for_i(i):
    return max(i / TRADING_DAYS_PER_YEAR, 1 / TRADING_DAYS_PER_YEAR)

def bar_metrics(price_series, return_series, years_elapsed):
    vals = performance_metrics(price_series, return_series, years_elapsed)
    percent_vals = vals[:4]
    sharpe_val = vals[4]
    return percent_vals, sharpe_val, format_metric_text(vals), vals

# ============================================================
# Initial/final metrics and fixed axis ranges
# ============================================================

metric_labels = ["Total Return", "CAGR", "Ann Vol", "Max DD", "Sharpe"]

initial_i = max(2, MIN_REGRESSION_OBS)

frame_indices = list(range(initial_i, len(df), FRAME_STRIDE))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

max_frame = TRADING_DAYS_PER_YEAR * N_YEARS
frame_indices = [i for i in frame_indices if i <= max_frame]
if len(frame_indices) == 0 or frame_indices[-1] != max_frame:
    frame_indices.append(max_frame)
frame_indices = [i for i in frame_indices if initial_i <= i <= max_frame]

market_metrics = performance_metrics(df["Market"], df["Market Return"], N_YEARS)
sentiment_metrics = performance_metrics(df["Sentiment"], df["Sentiment Return"], N_YEARS)

final_beta, final_intercept, realized_corr, final_r2 = regression_stats(
    returns_df["Market Return"],
    returns_df["Sentiment Return"],
)
final_ann_intercept = annualize_daily_intercept(final_intercept)
final_relative = df["Relative Sentiment vs Market"].iloc[-1]
realized_shock_corr = np.corrcoef(z_market, z_sentiment)[0, 1]

price_range = padded_range(np.r_[df["Market"].values, df["Sentiment"].values], 0.08)
market_dd_range = drawdown_range(df["Market Drawdown"])
sentiment_dd_range = drawdown_range(df["Sentiment Drawdown"])

# ===== Static (terminal) bar/Sharpe axis limits, and static bars for all frames =====

# Compute terminal bar values/texts for both Market and Sentiment only once
mkt_barvals, mkt_sharpeval, mkt_text, _ = bar_metrics(
    df["Market"],
    df["Market Return"],
    N_YEARS,
)
sent_barvals, sent_sharpeval, sent_text, _ = bar_metrics(
    df["Sentiment"],
    df["Sentiment Return"],
    N_YEARS,
)

def static_return_lim(barvals):
    maxval = np.max(barvals)
    minval = np.min(barvals)
    high = maxval + 0.1 * abs(maxval if maxval != 0 else 1)
    low = minval - 0.1 * abs(minval if minval != 0 else 1)
    if np.isclose(maxval, minval):
        high = maxval + 0.12 * abs(maxval if maxval != 0 else 1)
        low = minval - 0.12 * abs(minval if minval != 0 else 1)
    return [low, high]

def static_sharpe_lim(val):
    return [val - 0.25, val + 0.25]

mkt_return_axis_range = static_return_lim(mkt_barvals)
sent_return_axis_range = static_return_lim(sent_barvals)
mkt_sharpe_axis_range = static_sharpe_lim(mkt_sharpeval)
sent_sharpe_axis_range = static_sharpe_lim(sent_sharpeval)

x_min = float(returns_df["Market Return"].min())
x_max = float(returns_df["Market Return"].max())
x_pad = max((x_max - x_min) * 0.15, 0.005)
scatter_x_range = [x_min - x_pad, x_max + x_pad]

y_min = float(returns_df["Sentiment Return"].min())
y_max = float(returns_df["Sentiment Return"].max())
y_pad = max((y_max - y_min) * 0.15, 0.005)
scatter_y_range = [y_min - y_pad, y_max + y_pad]

reg_x = np.array(scatter_x_range)
text_x = scatter_x_range[0] + 0.05 * (scatter_x_range[1] - scatter_x_range[0])
text_y = scatter_y_range[1] - 0.10 * (scatter_y_range[1] - scatter_y_range[0])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
market_color = "#00d4ff"
sentiment_color = "#b940fe"
scatter_color = "rgba(224,224,224,0.68)"
regression_color = "#ffaa33"
baseline_color = "#777777"
drawdown_line_red = "#ff2222"
drawdown_fill_red = "rgba(255,60,60,0.24)"
sharpe_color = "#e865e8"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

# Adjust the row_heights so that there is more padding under the first (top) row.
# We decrease the height for row 2 and row 3 a bit, so row 1 gets relatively more space above the middle row.

fig = make_subplots(
    rows=3,
    cols=2,
    row_heights=[0.46, 0.26, 0.18],
    column_widths=[0.52, 0.48],
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
    subplot_titles=(
        "Cumulative Performance",
        "Daily Return Scatter with Expanding OLS Fit",
        "Market Drawdown",
        "Sentiment Drawdown",
        "Market Performance Metrics",
        "Sentiment Performance Metrics",
    ),
    specs=[
        [{}, {}],
        [{}, {}],
        [{"secondary_y": True}, {"secondary_y": True}],
    ],
)

initial_slice = df.iloc[: initial_i + 1]
initial_returns = returns_df.iloc[:initial_i]

initial_beta, initial_intercept, initial_corr, initial_r2 = regression_stats(
    initial_returns["Market Return"],
    initial_returns["Sentiment Return"],
)
initial_reg_y = initial_intercept + initial_beta * reg_x

# Top-left: cumulative performance
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Market"],
        mode="lines",
        line=dict(color=market_color, width=3),
        name="Market",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Market: %{y:.2f}<extra></extra>",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Sentiment"],
        mode="lines",
        line=dict(color=sentiment_color, width=3),
        name="Sentiment",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Sentiment: %{y:.2f}<extra></extra>",
    ),
    row=1, col=1,
)

# Top-right: animated scatter
fig.add_trace(
    go.Scatter(
        x=initial_returns["Market Return"],
        y=initial_returns["Sentiment Return"],
        mode="markers",
        marker=dict(size=6, color=scatter_color, line=dict(width=0)),
        customdata=initial_returns["date"],
        name="Return Scatter",
        showlegend=False,
        hovertemplate=(
            "Date: %{customdata|%Y-%m-%d}<br>"
            "Market Return: %{x:.2%}<br>"
            "Sentiment Return: %{y:.2%}<extra></extra>"
        ),
    ),
    row=1, col=2,
)

# Expanding OLS regression line.
fig.add_trace(
    go.Scatter(
        x=reg_x,
        y=initial_reg_y,
        mode="lines",
        line=dict(color=regression_color, width=3, dash="dash"),
        name="Expanding OLS Fit",
        showlegend=False,
        hovertemplate="Expanding OLS fit<extra></extra>",
    ),
    row=1, col=2,
)

# Middle row: separate drawdowns
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Market Drawdown"],
        mode="lines",
        line=dict(color=drawdown_line_red, width=2),
        fill="tozeroy",
        fillcolor=drawdown_fill_red,
        showlegend=False,
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Market Drawdown: %{y:.2%}<extra></extra>",
    ),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Sentiment Drawdown"],
        mode="lines",
        line=dict(color=drawdown_line_red, width=2),
        fill="tozeroy",
        fillcolor=drawdown_fill_red,
        showlegend=False,
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Sentiment Drawdown: %{y:.2%}<extra></extra>",
    ),
    row=2, col=2,
)

# Bottom row: STATIC bar charts for ALL frames
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=mkt_barvals,
        marker=dict(color=[market_color] * 4, opacity=0.84),
        showlegend=False,
        text=mkt_text[:4],
        textposition="auto",
        name="Market Metrics",
        customdata=mkt_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=1, secondary_y=False,
)
fig.add_trace(
    go.Bar(
        x=[metric_labels[4]],
        y=[mkt_sharpeval],
        marker=dict(color=[sharpe_color], opacity=0.85),
        showlegend=False,
        text=[mkt_text[4]],
        textposition="auto",
        name="Market Sharpe",
        customdata=[mkt_text[4]],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=1, secondary_y=True,
)
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=sent_barvals,
        marker=dict(color=[sentiment_color] * 4, opacity=0.84),
        showlegend=False,
        text=sent_text[:4],
        textposition="auto",
        name="Sentiment Metrics",
        customdata=sent_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=2, secondary_y=False,
)
fig.add_trace(
    go.Bar(
        x=[metric_labels[4]],
        y=[sent_sharpeval],
        marker=dict(color=[sharpe_color], opacity=0.85),
        showlegend=False,
        text=[sent_text[4]],
        textposition="auto",
        name="Sentiment Sharpe",
        customdata=[sent_text[4]],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3, col=2, secondary_y=True,
)

# Reference lines
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_vline(x=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=1)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=1, secondary_y=False)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=2, secondary_y=False)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

for idx, i in enumerate(frame_indices):
    frame_name = f"f{i}"
    current_slice = df.iloc[: i + 1]
    current_returns = returns_df.iloc[:i]
    years_elapsed = elapsed_years_for_i(i)

    # Regression/statistics for expanding scatter only, bars are static
    beta, intercept, corr, r2 = regression_stats(
        current_returns["Market Return"],
        current_returns["Sentiment Return"],
    )
    reg_y = intercept + beta * reg_x

    # Always use the terminal bar values/text for both market and sentiment:
    frame_data = [
        go.Scatter(x=current_slice["date"], y=current_slice["Market"]),
        go.Scatter(x=current_slice["date"], y=current_slice["Sentiment"], line=dict(color=sentiment_color, width=3)),
        go.Scatter(
            x=current_returns["Market Return"],
            y=current_returns["Sentiment Return"],
            customdata=current_returns["date"],
        ),
        go.Scatter(x=reg_x, y=reg_y),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Market Drawdown"],
            line=dict(color=drawdown_line_red, width=2),
            fill="tozeroy",
            fillcolor=drawdown_fill_red,
        ),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Sentiment Drawdown"],
            line=dict(color=drawdown_line_red, width=2),
            fill="tozeroy",
            fillcolor=drawdown_fill_red,
        ),
        # Market metric bars (static/terminal):
        go.Bar(
            x=metric_labels[:4],
            y=mkt_barvals,
            marker=dict(color=[market_color] * 4, opacity=0.84),
            text=mkt_text[:4],
            textposition="auto",
            customdata=mkt_text[:4],
        ),
        go.Bar(
            x=[metric_labels[4]],
            y=[mkt_sharpeval],
            marker=dict(color=[sharpe_color], opacity=0.85),
            text=[mkt_text[4]],
            textposition="auto",
            customdata=[mkt_text[4]],
        ),
        # Sentiment metric bars (always use terminal):
        go.Bar(
            x=metric_labels[:4],
            y=sent_barvals,
            marker=dict(color=[sentiment_color] * 4, opacity=0.84),
            text=sent_text[:4],
            textposition="auto",
            customdata=sent_text[:4],
        ),
        go.Bar(
            x=[metric_labels[4]],
            y=[sent_sharpeval],
            marker=dict(color=[sharpe_color], opacity=0.85),
            text=[sent_text[4]],
            textposition="auto",
            customdata=[sent_text[4]],
        ),
    ]

    frames.append(
        go.Frame(
            data=frame_data,
            traces=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],  # total 10 traces (was 11)
            name=frame_name,
            layout=dict(
                yaxis5=dict(range=mkt_return_axis_range),
                yaxis6=dict(range=mkt_sharpe_axis_range),
                yaxis7=dict(range=sent_return_axis_range),
                yaxis8=dict(range=sent_sharpe_axis_range),
            ),
        )
    )

    elapsed_years = i / TRADING_DAYS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Orthogonal Risk Premia: Market vs Sentiment",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=145, b=150, r=50, l=75),
    legend=dict(
        orientation="v",
        x=0,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right",
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=13))

# Cumulative performance axes
fig.update_xaxes(
    axis_style,
    row=1, col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date",
)
fig.update_yaxes(axis_style, row=1, col=1, range=price_range, title_text="Index Value")

# Scatter axes
fig.update_xaxes(
    axis_style,
    row=1, col=2,
    range=scatter_x_range,
    title_text="Market Daily Return",
    tickformat=".1%",
)
fig.update_yaxes(
    axis_style,
    row=1, col=2,
    range=scatter_y_range,
    title_text="Sentiment Daily Return",
    tickformat=".1%",
)

# Drawdown charts
fig.update_xaxes(
    axis_style,
    row=2, col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="",
)
fig.update_yaxes(
    axis_style,
    row=2, col=1,
    range=market_dd_range,
    title_text="Drawdown",
    tickformat=".0%",
    color=drawdown_line_red,
    linecolor=drawdown_line_red,
    tickfont=dict(color="#ffbbbb"),
)

fig.update_xaxes(
    axis_style,
    row=2, col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="",
)
fig.update_yaxes(
    axis_style,
    row=2, col=2,
    range=sentiment_dd_range,
    title_text="Drawdown",
    tickformat=".0%",
    color=drawdown_line_red,
    linecolor=drawdown_line_red,
    tickfont=dict(color="#ffbbbb"),
)

# Performance metric bars: STATIC y is percent, STATIC y is Sharpe.
fig.update_xaxes(axis_style, row=3, col=1, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3, col=1,
    range=mkt_return_axis_range,
    title_text="Percent values",
    secondary_y=False,
)
# Market (left) twin y: Sharpe Ratio label removed as before
fig.update_yaxes(
    dict(
        title_text="",  # No label for the market Sharpe twin axis
        showgrid=False,
        showline=True,
        zeroline=False,
        tickfont=dict(color=sharpe_color),
        title_font=dict(color=sharpe_color),
        range=mkt_sharpe_axis_range,
    ),
    row=3, col=1, secondary_y=True,
)

fig.update_xaxes(axis_style, row=3, col=2, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3, col=2,
    range=sent_return_axis_range,
    title_text="Percent values",
    secondary_y=False,
)
fig.update_yaxes(
    dict(
        title_text="Sharpe Ratio",
        showgrid=False,
        showline=True,
        zeroline=False,
        tickfont=dict(color=sharpe_color),
        title_font=dict(color=sharpe_color),
        range=sent_sharpe_axis_range,
    ),
    row=3, col=2, secondary_y=True,
)

# ============================================================
# Save / show
# ============================================================
fig.show()


###### ______________________________________________________________________________________________________________________________________

 

##### ⚾ Allocating Risk is Stepping Up to the Plate

We don't get to pick the pitch, but we know the pitcher (market regime).

He likes curveballs, it doesn't mean he won't throw a fastball.  

Statistics gives us an edge, we're ready for his favorite pitch, but aren't only going to commit to that one.

**Example:** My (quant, discretionary, whatever) model says this year is a bear market, so I'm going to step up to the plate and position myself for this fastball.

A bull market is then a curveball, but if it is in fact that fastball I'm going to crack it out of the park.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 42
np.random.seed(SEED)

N_YEARS = 2
TRADING_DAYS_PER_YEAR = 252
N_DAYS = N_YEARS * TRADING_DAYS_PER_YEAR
DT = 1 / TRADING_DAYS_PER_YEAR

INITIAL_VALUE = 100
START_DATE = "2026-01-01"
RISK_FREE_RATE = 0.00

# ------------------------------------------------------------
# Two-regime setup
# ------------------------------------------------------------
# Regime 1: bull market. The strategy has a small positive market beta but
# deliberately negative alpha, so it underperforms the market in the rally.
# Regime 2: bear/crisis market. The strategy beta flips negative and has
# crisis convexity, so it becomes negatively correlated to the market and
# shows positive regression alpha during the crash.

BULL_DAYS = TRADING_DAYS_PER_YEAR
BEAR_DAYS = N_DAYS - BULL_DAYS
CRASH_THRESHOLD = 0.006
ORTHOGONALIZE_IDIOSYNCRATIC_BY_REGIME = True

REGIME_PARAMS = {
    "Bull": {
        "market_mu": 0.18,
        "market_sigma": 0.13,
        "strategy_beta": 0.15,
        "strategy_alpha": -0.04,
        "strategy_idio_sigma": 0.08,
        "strategy_crash_convexity": 0.00,
        "description": "Bull: market rallies; strategy intentionally lags",
    },
    "Bear/Crisis": {
        "market_mu": -0.32,
        "market_sigma": 0.34,
        "strategy_beta": -0.65,
        "strategy_alpha": 0.06,
        "strategy_idio_sigma": 0.10,
        "strategy_crash_convexity": 0.15,
        "description": "Bear/Crisis: market sells off; strategy flips short-beta and generates alpha",
    },
}

FRAME_STRIDE = 5
FRAME_DURATION = 20
MIN_REGRESSION_OBS = 20

OUTPUT_HTML = "market_vs_trading_strategy_two_regimes.html"
SHOW_FIG = False

# ============================================================
# Simulate market and regime-aware trading strategy
# ============================================================

dates = pd.bdate_range(start=START_DATE, periods=N_DAYS + 1)
return_regime = np.array(["Bull"] * BULL_DAYS + ["Bear/Crisis"] * BEAR_DAYS)
price_regime = np.r_[["Start"], return_regime]

market_mu = np.array([REGIME_PARAMS[r]["market_mu"] for r in return_regime])
market_sigma = np.array([REGIME_PARAMS[r]["market_sigma"] for r in return_regime])
strategy_beta_input = np.array([REGIME_PARAMS[r]["strategy_beta"] for r in return_regime])
strategy_alpha_input = np.array([REGIME_PARAMS[r]["strategy_alpha"] for r in return_regime])
strategy_idio_sigma = np.array([REGIME_PARAMS[r]["strategy_idio_sigma"] for r in return_regime])
strategy_crash_convexity = np.array([REGIME_PARAMS[r]["strategy_crash_convexity"] for r in return_regime])

z_market = np.random.normal(size=N_DAYS)
z_strategy_raw = np.random.normal(size=N_DAYS)

# Make the strategy's idiosyncratic component independent of the market shock
# inside each regime. The realized beta/correlation is then driven by the
# explicit regime beta, not by accidental random co-movement.
if ORTHOGONALIZE_IDIOSYNCRATIC_BY_REGIME:
    z_strategy = np.empty(N_DAYS)
    for regime_name in REGIME_PARAMS:
        mask = return_regime == regime_name
        z_m = z_market[mask] - z_market[mask].mean()
        z_s = z_strategy_raw[mask] - z_strategy_raw[mask].mean()
        projection = (np.dot(z_s, z_m) / np.dot(z_m, z_m)) * z_m
        orthogonal = z_s - projection
        z_strategy[mask] = orthogonal / orthogonal.std(ddof=0)
else:
    z_strategy = z_strategy_raw

market_log_return = (
    (market_mu - 0.5 * market_sigma**2) * DT
    + market_sigma * np.sqrt(DT) * z_market
)
market_return = np.exp(market_log_return) - 1

crisis_payoff = strategy_crash_convexity * np.maximum(-market_return - CRASH_THRESHOLD, 0)
strategy_return = (
    strategy_alpha_input * DT
    + strategy_beta_input * market_return
    + strategy_idio_sigma * np.sqrt(DT) * z_strategy
    + crisis_payoff
)
strategy_return = np.clip(strategy_return, -0.50, 0.50)

market = INITIAL_VALUE * np.r_[1, np.cumprod(1 + market_return)]
strategy = INITIAL_VALUE * np.r_[1, np.cumprod(1 + strategy_return)]

# Price/index DataFrame.
df = pd.DataFrame(
    {
        "date": dates,
        "Regime": price_regime,
        "Market": market,
        "Trading Strategy": strategy,
    }
)

df["Market Return"] = df["Market"].pct_change()
df["Strategy Return"] = df["Trading Strategy"].pct_change()

df["Market Drawdown"] = df["Market"] / df["Market"].cummax() - 1
df["Strategy Drawdown"] = df["Trading Strategy"] / df["Trading Strategy"].cummax() - 1

df["Relative Strategy vs Market"] = df["Trading Strategy"] / df["Market"] - 1
returns_df = df.dropna().reset_index(drop=True)
regime_change_date = df["date"].iloc[BULL_DAYS]

# ============================================================
# Helpers
# ============================================================

def max_drawdown(series):
    return (series / series.cummax() - 1).min()

def performance_metrics(price_series, return_series, years_elapsed=None):
    price_series = price_series.dropna()
    return_series = return_series.dropna()
    if years_elapsed is None:
        years_elapsed = max(len(return_series) / TRADING_DAYS_PER_YEAR, 1 / TRADING_DAYS_PER_YEAR)
    else:
        years_elapsed = max(years_elapsed, 1 / TRADING_DAYS_PER_YEAR)
    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    cagr = (price_series.iloc[-1] / price_series.iloc[0]) ** (1 / years_elapsed) - 1
    ann_vol = return_series.std(ddof=0) * np.sqrt(TRADING_DAYS_PER_YEAR) if len(return_series) > 1 else 0.0
    mdd = max_drawdown(price_series)
    # Only return return-based metrics (not Sharpe)
    return np.array(
        [
            total_return * 100,
            cagr * 100,
            ann_vol * 100,
            abs(mdd) * 100,
            # Sharpe removed
        ]
    )

def format_metric_text(values):
    # Only format the four percent metrics (Sharpe removed)
    return [
        f"{values[0]:.1f}%",
        f"{values[1]:.1f}%",
        f"{values[2]:.1f}%",
        f"{values[3]:.1f}%",
        # Sharpe text removed
    ]

def padded_range(values, pad_fraction=0.08, min_pad=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]

def drawdown_range(drawdown_series):
    dd_min = float(drawdown_series.min())
    lower = min(dd_min * 1.15, -0.02)
    return [lower, 0.02]

def regression_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.isclose(np.std(x), 0):
        intercept = float(np.mean(y)) if len(y) else 0.0
        return 0.0, intercept, 0.0, 0.0
    beta, intercept = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1]
    r2 = corr**2
    return float(beta), float(intercept), float(corr), float(r2)

def regression_stats_for_regime(regime_name):
    mask = returns_df["Regime"].eq(regime_name)
    return regression_stats(
        returns_df.loc[mask, "Market Return"],
        returns_df.loc[mask, "Strategy Return"],
    )

def annualize_daily_intercept(daily_intercept):
    if daily_intercept <= -1:
        return -1.0
    return (1 + daily_intercept) ** TRADING_DAYS_PER_YEAR - 1

def regression_text(beta, intercept, corr, r2):
    ann_intercept = annualize_daily_intercept(intercept)
    return (
        f"β = {beta:.2f}<br>"
        f"Corr = {corr:.2f} · R² = {r2:.2f}<br>"
        f"Ann. α = {ann_intercept:.1%}"
    )

def elapsed_years_for_i(i):
    return max(i / TRADING_DAYS_PER_YEAR, 1 / TRADING_DAYS_PER_YEAR)

def bar_metrics(price_series, return_series, years_elapsed):
    vals = performance_metrics(price_series, return_series, years_elapsed)
    percent_vals = vals[:4]
    # Sharpe removed entirely
    return percent_vals, format_metric_text(vals), vals

def static_return_lim(barvals):
    maxval = np.max(barvals)
    minval = np.min(barvals)
    high = maxval + 0.12 * abs(maxval if maxval != 0 else 1)
    low = minval - 0.12 * abs(minval if minval != 0 else 1)
    if np.isclose(maxval, minval):
        high = maxval + 0.15 * abs(maxval if maxval != 0 else 1)
        low = minval - 0.15 * abs(minval if minval != 0 else 1)
    return [low, high]

def regime_marker_colors(regime_series):
    return np.where(
        pd.Series(regime_series).eq("Bear/Crisis"),
        bear_scatter_color,
        bull_scatter_color,
    )

# ============================================================
# Initial/final metrics and fixed axis ranges
# ============================================================

metric_labels = ["Total Return", "CAGR", "Ann Vol", "Max DD"]
# Remove Sharpe from all metrics

initial_i = max(2, MIN_REGRESSION_OBS)

frame_indices = list(range(initial_i, len(df), FRAME_STRIDE))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

max_frame = TRADING_DAYS_PER_YEAR * N_YEARS
frame_indices = [i for i in frame_indices if i <= max_frame]
if len(frame_indices) == 0 or frame_indices[-1] != max_frame:
    frame_indices.append(max_frame)
frame_indices = [i for i in frame_indices if initial_i <= i <= max_frame]

market_metrics = performance_metrics(df["Market"], df["Market Return"], N_YEARS)
strategy_metrics = performance_metrics(df["Trading Strategy"], df["Strategy Return"], N_YEARS)

final_beta, final_intercept, realized_corr, final_r2 = regression_stats(
    returns_df["Market Return"],
    returns_df["Strategy Return"],
)
bull_beta, bull_intercept, bull_corr, bull_r2 = regression_stats_for_regime("Bull")
bear_beta, bear_intercept, bear_corr, bear_r2 = regression_stats_for_regime("Bear/Crisis")

final_ann_alpha = annualize_daily_intercept(final_intercept)
bull_ann_alpha = annualize_daily_intercept(bull_intercept)
bear_ann_alpha = annualize_daily_intercept(bear_intercept)
final_relative = df["Relative Strategy vs Market"].iloc[-1]

price_range = padded_range(np.r_[df["Market"].values, df["Trading Strategy"].values], 0.08)
market_dd_range = drawdown_range(df["Market Drawdown"])
strategy_dd_range = drawdown_range(df["Strategy Drawdown"])

mkt_barvals, mkt_text, _ = bar_metrics(
    df["Market"],
    df["Market Return"],
    N_YEARS,
)
strat_barvals, strat_text, _ = bar_metrics(
    df["Trading Strategy"],
    df["Strategy Return"],
    N_YEARS,
)

mkt_return_axis_range = static_return_lim(mkt_barvals)
strat_return_axis_range = static_return_lim(strat_barvals)

x_min = float(returns_df["Market Return"].min())
x_max = float(returns_df["Market Return"].max())
x_pad = max((x_max - x_min) * 0.15, 0.005)
scatter_x_range = [x_min - x_pad, x_max + x_pad]

y_min = float(returns_df["Strategy Return"].min())
y_max = float(returns_df["Strategy Return"].max())
y_pad = max((y_max - y_min) * 0.15, 0.005)
scatter_y_range = [y_min - y_pad, y_max + y_pad]

reg_x = np.array(scatter_x_range)
text_x = scatter_x_range[0] + 0.05 * (scatter_x_range[1] - scatter_x_range[0])
text_y = scatter_y_range[1] - 0.10 * (scatter_y_range[1] - scatter_y_range[0])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
market_color = "#00d4ff"
strategy_color = "#ff003c"  # neon red strategy path
strategy_glow_color = "rgba(255, 0, 60, 0.22)"
bull_scatter_color = "rgba(0, 212, 255, 0.42)"
bear_scatter_color = "rgba(255, 0, 60, 0.76)"
regression_color = "#ffaa33"
baseline_color = "#777777"
market_drawdown_fill = "rgba(0, 212, 255, 0.16)"
strategy_drawdown_fill = "rgba(255, 0, 60, 0.22)"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=3,
    cols=2,
    row_heights=[0.46, 0.32, 0.22],
    column_widths=[0.52, 0.48],
    horizontal_spacing=0.08,
    vertical_spacing=0.14,  # <<---- INCREASED VERTICAL SPACING (default was 0.09)
    subplot_titles=(
        "",  # Remove all subtitles for main and subplots
        "",
        "",
        "",
        "",
        "",
    ),
    specs=[
        [{}, {}],
        [{}, {}],
        [{}, {}],
    ],
)

initial_slice = df.iloc[: initial_i + 1]
initial_returns = returns_df.iloc[:initial_i]
initial_customdata = initial_returns[["date", "Regime"]].to_numpy()
initial_marker_colors = regime_marker_colors(initial_returns["Regime"])

initial_beta, initial_intercept, initial_corr, initial_r2 = regression_stats(
    initial_returns["Market Return"],
    initial_returns["Strategy Return"],
)
initial_reg_y = initial_intercept + initial_beta * reg_x

# Top-left: cumulative performance
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Market"],
        mode="lines",
        line=dict(color=market_color, width=3),
        name="Market",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Market: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Neon-red glow under the strategy path.
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Trading Strategy"],
        mode="lines",
        line=dict(color=strategy_glow_color, width=11),
        name="Trading Strategy Glow",
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Trading Strategy"],
        mode="lines",
        line=dict(color=strategy_color, width=4),
        name="Trading Strategy",
        showlegend=True,
        legendgroup="paths",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Trading Strategy: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Top-right: animated scatter
fig.add_trace(
    go.Scatter(
        x=initial_returns["Market Return"],
        y=initial_returns["Strategy Return"],
        mode="markers",
        marker=dict(size=6, color=initial_marker_colors, line=dict(width=0)),
        customdata=initial_customdata,
        name="Return Scatter",
        showlegend=False,
        hovertemplate=(
            "Date: %{customdata[0]|%Y-%m-%d}<br>"
            "Regime: %{customdata[1]}<br>"
            "Market Return: %{x:.2%}<br>"
            "Strategy Return: %{y:.2%}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Expanding OLS regression line.
fig.add_trace(
    go.Scatter(
        x=reg_x,
        y=initial_reg_y,
        mode="lines",
        line=dict(color=regression_color, width=3, dash="dash"),
        name="Expanding OLS Fit",
        showlegend=False,
        hovertemplate="Expanding OLS fit<extra></extra>",
    ),
    row=1,
    col=2,
)

# REMOVE the regression text overlay (delete the following Scatter trace)
# fig.add_trace(
#     go.Scatter(
#         x=[text_x],
#         y=[text_y],
#         mode="text",
#         text=[regression_text(initial_beta, initial_intercept, initial_corr, initial_r2)],
#         textfont=dict(color=off_white, size=13),
#         textposition="middle left",
#         showlegend=False,
#         hoverinfo="skip",
#     ),
#     row=1,
#     col=2,
# )

# Middle row: separate drawdowns
fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Market Drawdown"],
        mode="lines",
        line=dict(color=market_color, width=2),
        fill="tozeroy",
        fillcolor=market_drawdown_fill,
        showlegend=False,
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Market Drawdown: %{y:.2%}<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=initial_slice["date"],
        y=initial_slice["Strategy Drawdown"],
        mode="lines",
        line=dict(color=strategy_color, width=2),
        fill="tozeroy",
        fillcolor=strategy_drawdown_fill,
        showlegend=False,
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Strategy Drawdown: %{y:.2%}<extra></extra>",
    ),
    row=2,
    col=2,
)

# Bottom row: terminal metric bars for all frames.
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=mkt_barvals,
        marker=dict(color=[market_color] * 4, opacity=0.84),
        showlegend=False,
        text=mkt_text[:4],
        textposition="auto",
        name="Market Metrics",
        customdata=mkt_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=metric_labels[:4],
        y=strat_barvals,
        marker=dict(color=[strategy_color] * 4, opacity=0.86),
        showlegend=False,
        text=strat_text[:4],
        textposition="auto",
        name="Strategy Metrics",
        customdata=strat_text[:4],
        hovertemplate="%{x}: %{customdata}<extra></extra>",
    ),
    row=3,
    col=2,
)

# Reference lines.
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_vline(x=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=1, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=1)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=2, col=2)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=1)
fig.add_hline(y=0, line=dict(color=baseline_color, width=1, dash="dash"), opacity=0.65, row=3, col=2)

# Regime separator on date-based panels.
for r, c in [(1, 1), (2, 1), (2, 2)]:
    fig.add_vline(
        x=regime_change_date,
        line=dict(color="rgba(255,255,255,0.46)", width=1, dash="dot"),
        row=r,
        col=c,
    )

fig.add_annotation(
    x=regime_change_date,
    y=price_range[1],
    xref="x",
    yref="y",
    text="Bear/Crisis starts",
    showarrow=True,
    arrowhead=2,
    ax=35,
    ay=-30,
    font=dict(color=off_white, size=12),
    arrowcolor="rgba(255,255,255,0.55)",
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

for idx, i in enumerate(frame_indices):
    frame_name = f"f{i}"
    current_slice = df.iloc[: i + 1]
    current_returns = returns_df.iloc[:i]
    current_customdata = current_returns[["date", "Regime"]].to_numpy()
    current_marker_colors = regime_marker_colors(current_returns["Regime"])

    # Expanding regression/statistics; metric bars stay terminal/static.
    beta, intercept, corr, r2 = regression_stats(
        current_returns["Market Return"],
        current_returns["Strategy Return"],
    )
    reg_y = intercept + beta * reg_x

    frame_data = [
        go.Scatter(x=current_slice["date"], y=current_slice["Market"]),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Trading Strategy"],
            line=dict(color=strategy_glow_color, width=11),
        ),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Trading Strategy"],
            line=dict(color=strategy_color, width=4),
        ),
        go.Scatter(
            x=current_returns["Market Return"],
            y=current_returns["Strategy Return"],
            marker=dict(size=6, color=current_marker_colors, line=dict(width=0)),
            customdata=current_customdata,
        ),
        go.Scatter(x=reg_x, y=reg_y),
        # REMOVE the animated regression text overlay
        # go.Scatter(
        #     x=[text_x],
        #     y=[text_y],
        #     text=[regression_text(beta, intercept, corr, r2)],
        # ),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Market Drawdown"],
            line=dict(color=market_color, width=2),
            fill="tozeroy",
            fillcolor=market_drawdown_fill,
        ),
        go.Scatter(
            x=current_slice["date"],
            y=current_slice["Strategy Drawdown"],
            line=dict(color=strategy_color, width=2),
            fill="tozeroy",
            fillcolor=strategy_drawdown_fill,
        ),
        go.Bar(
            x=metric_labels[:4],
            y=mkt_barvals,
            marker=dict(color=[market_color] * 4, opacity=0.84),
            text=mkt_text[:4],
            textposition="auto",
            customdata=mkt_text[:4],
        ),
        go.Bar(
            x=metric_labels[:4],
            y=strat_barvals,
            marker=dict(color=[strategy_color] * 4, opacity=0.86),
            text=strat_text[:4],
            textposition="auto",
            customdata=strat_text[:4],
        ),
    ]

    # For correct trace references in go.Frame (now traces=range(9) instead of 10)
    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(9)),
            name=frame_name,
            layout=dict(
                yaxis5=dict(range=mkt_return_axis_range),
                yaxis6=dict(range=strat_return_axis_range),
            ),
        )
    )

    elapsed_years = i / TRADING_DAYS_PER_YEAR
    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                    "fromcurrent": True,
                },
            ],
            "label": f"{elapsed_years:.1f}Y",
            "method": "animate",
        }
    )

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Market vs Trading Strategy: Bull Underperformance, Crisis Alpha"
            # Removed all subtitle/sup text (no <sup>, no extra stats lines)
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=165, b=150, r=50, l=75),
    legend=dict(
        orientation="v",
        x=0,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION, "redraw": False},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 14, "color": off_white},
                "prefix": "Through: ",
                "visible": True,
                "xanchor": "right",
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 50},
            "len": 0.85,
            "x": 0.15,
            "y": 0,
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=13))

# Cumulative performance axes.
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date",
)
fig.update_yaxes(axis_style, row=1, col=1, range=price_range, title_text="Index Value")

# Scatter axes.
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=scatter_x_range,
    title_text="Market Daily Return",
    tickformat=".1%",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=scatter_y_range,
    title_text="Strategy Daily Return",
    tickformat=".1%",
)

# Drawdown charts.
fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="",
)
fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    range=market_dd_range,
    title_text="Drawdown",
    tickformat=".0%",
    color=market_color,
    linecolor=market_color,
    tickfont=dict(color="#b7f4ff"),
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="",
)
fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    range=strategy_dd_range,
    title_text="Drawdown",
    tickformat=".0%",
    color=strategy_color,
    linecolor=strategy_color,
    tickfont=dict(color="#ffb7c5"),
)

# Performance metric bars: only percent values, no Sharpe.
fig.update_xaxes(axis_style, row=3, col=1, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3,
    col=1,
    range=mkt_return_axis_range,
    title_text="Percent values",
)
fig.update_xaxes(axis_style, row=3, col=2, title_text="", tickangle=0)
fig.update_yaxes(
    axis_style,
    row=3,
    col=2,
    range=strat_return_axis_range,
    title_text="Percent values",
)

# ============================================================
# Save / show
# ============================================================

fig.show()

###### ______________________________________________________________________________________________________________________________________

 

##### 🏛️ Alpha vs. Allocation

**Remark:** Experienced allocation leads to alpha.

Generating alpha is like hitting a home run, academics say it's a statistical anomaly, but you were in the position to swing at the pitch. 

You don't hit a home run every pitch.  If you did you would likely have a secretive strategy (inefficiency, think like you know their hand signals)

The inefficiency can go away (they can change their hand signals whenever they want) but for the time being, you are outpreforming and nobody knows why

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 42
np.random.seed(SEED)

TRADING_DAYS_PER_YEAR = 252
DAYS_PER_REGIME = TRADING_DAYS_PER_YEAR
DT = 1 / TRADING_DAYS_PER_YEAR

INITIAL_VALUE = 100
START_DATE = "2026-01-01"

FRAME_STRIDE = 5
FRAME_DURATION = 22
MIN_REGRESSION_OBS = 20

OUTPUT_HTML = "secret_strategy_regime_stack.html"
SHOW_FIG = False

# ------------------------------------------------------------
# Three independent market regimes
# ------------------------------------------------------------
# The Secret Strategy is intentionally constructed to be nearly orthogonal
# to each market regime.  It keeps a positive drift and low idiosyncratic
# volatility in Bull, Sideways, and Bear markets, so each regime's OLS fit
# should show roughly zero beta/slope while the path compounds consistently.

REGIME_CONFIGS = {
    "Bull": {
        "market_mu": 0.18,
        "market_sigma": 0.13,
        "strategy_mu": 0.115,
        "strategy_sigma": 0.060,
        "strategy_beta": 0.00,
        "description": "Market rallies; Secret Strategy compounds independently",
    },
    "Sideways": {
        "market_mu": 0.00,
        "market_sigma": 0.16,
        "strategy_mu": 0.110,
        "strategy_sigma": 0.055,
        "strategy_beta": 0.00,
        "description": "Market chops; Secret Strategy keeps steady positive drift",
    },
    "Bear": {
        "market_mu": -0.24,
        "market_sigma": 0.30,
        "strategy_mu": 0.120,
        "strategy_sigma": 0.065,
        "strategy_beta": 0.00,
        "description": "Market sells off; Secret Strategy remains market neutral",
    },
}

# ============================================================
# Simulation helpers
# ============================================================

def orthogonalize(y, x):
    """Remove the linear projection of y on x and standardize."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_centered = x - x.mean()
    y_centered = y - y.mean()

    denom = np.dot(x_centered, x_centered)
    if np.isclose(denom, 0):
        out = y_centered
    else:
        projection = (np.dot(y_centered, x_centered) / denom) * x_centered
        out = y_centered - projection

    std = out.std(ddof=0)
    if np.isclose(std, 0):
        return out
    return out / std


def simulate_regime(regime_name, config):
    dates = pd.bdate_range(start=START_DATE, periods=DAYS_PER_REGIME + 1)
    day = np.arange(DAYS_PER_REGIME + 1)

    z_market = np.random.normal(size=DAYS_PER_REGIME)
    z_market = (z_market - z_market.mean()) / z_market.std(ddof=0)
    z_strategy_raw = np.random.normal(size=DAYS_PER_REGIME)
    z_strategy = orthogonalize(z_strategy_raw, z_market)

    market_mu = config["market_mu"]
    market_sigma = config["market_sigma"]
    strategy_mu = config["strategy_mu"]
    strategy_sigma = config["strategy_sigma"]
    strategy_beta = config["strategy_beta"]

    market_log_return = (
        (market_mu - 0.5 * market_sigma**2) * DT
        + market_sigma * np.sqrt(DT) * z_market
    )
    market_return = np.exp(market_log_return) - 1

    # Market-neutral strategy return. The orthogonal shock makes the realized
    # regression slope very close to zero inside each regime.
    strategy_return = (
        strategy_mu * DT
        + strategy_beta * market_return
        + strategy_sigma * np.sqrt(DT) * z_strategy
    )
    strategy_return = np.clip(strategy_return, -0.50, 0.50)

    market = INITIAL_VALUE * np.r_[1, np.cumprod(1 + market_return)]
    strategy = INITIAL_VALUE * np.r_[1, np.cumprod(1 + strategy_return)]

    df = pd.DataFrame(
        {
            "date": dates,
            "day": day,
            "Regime": regime_name,
            "Market": market,
            "Secret Strategy": strategy,
        }
    )
    df["Market Return"] = df["Market"].pct_change()
    df["Strategy Return"] = df["Secret Strategy"].pct_change()
    df["Relative Secret Strategy vs Market"] = df["Secret Strategy"] / df["Market"] - 1
    return df


def regression_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.isclose(np.std(x), 0):
        intercept = float(np.mean(y)) if len(y) else 0.0
        return 0.0, intercept, 0.0, 0.0
    beta, intercept = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1]
    r2 = corr**2
    return float(beta), float(intercept), float(corr), float(r2)


def annualize_daily_intercept(daily_intercept):
    if daily_intercept <= -1:
        return -1.0
    return (1 + daily_intercept) ** TRADING_DAYS_PER_YEAR - 1


def regression_text(beta, intercept, corr, r2):
    ann_alpha = annualize_daily_intercept(intercept)
    return (
        f"β = {beta:.2f}<br>"
        f"Corr = {corr:.2f} · R² = {r2:.2f}<br>"
        f"Ann. α = {ann_alpha:.1%}"
    )


def padded_range(values, pad_fraction=0.08, min_pad=0.01):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]


def slice_returns(regime_df, i):
    # i is a price-row index, so returns through i are rows 1..i.
    return regime_df.iloc[1 : i + 1].copy()

# ============================================================
# Build data
# ============================================================

regime_names = list(REGIME_CONFIGS.keys())
regime_dfs = {
    regime_name: simulate_regime(regime_name, REGIME_CONFIGS[regime_name])
    for regime_name in regime_names
}

returns_by_regime = {
    regime_name: regime_dfs[regime_name].dropna().reset_index(drop=True)
    for regime_name in regime_names
}

initial_i = max(2, MIN_REGRESSION_OBS)
frame_indices = list(range(initial_i, DAYS_PER_REGIME + 1, FRAME_STRIDE))
if frame_indices[-1] != DAYS_PER_REGIME:
    frame_indices.append(DAYS_PER_REGIME)

# Common scatter axis ranges across all regimes for clean visual comparison.
all_market_returns = np.concatenate(
    [returns_by_regime[r]["Market Return"].values for r in regime_names]
)
all_strategy_returns = np.concatenate(
    [returns_by_regime[r]["Strategy Return"].values for r in regime_names]
)
scatter_x_range = padded_range(all_market_returns, pad_fraction=0.15, min_pad=0.005)
scatter_y_range = padded_range(all_strategy_returns, pad_fraction=0.18, min_pad=0.004)
reg_x = np.array(scatter_x_range)
text_x = scatter_x_range[0] + 0.05 * (scatter_x_range[1] - scatter_x_range[0])
text_y = scatter_y_range[1] - 0.10 * (scatter_y_range[1] - scatter_y_range[0])

price_ranges = {}
for regime_name in regime_names:
    d = regime_dfs[regime_name]
    price_ranges[regime_name] = padded_range(
        np.r_[d["Market"].values, d["Secret Strategy"].values],
        pad_fraction=0.08,
        min_pad=2.0,
    )

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
market_color = "#00d4ff"
strategy_color = "#39ff14"       # neon green path
strategy_glow_color = "rgba(57, 255, 20, 0.24)"
scatter_color = "rgba(224,224,224,0.62)"
regression_color = "#ffaa33"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

# The alpha (annualized intercept) shown in the grid column subtitles is now computed from the
# regression using *all* the returns collected throughout the full simulation in each regime.
# This will reflect the terminal empirical alpha, not the alpha at an initial subsample.

subplot_titles = []
terminal_regression_anns = {}  # Save terminal annualized alpha per regime for the subplot titles
for regime_name in regime_names:
    d = regime_dfs[regime_name]
    terminal_returns = slice_returns(d, DAYS_PER_REGIME)  # slice all returns up to the final period
    _, terminal_intercept, _, _ = regression_stats(
        terminal_returns["Market Return"], terminal_returns["Strategy Return"]
    )
    ann_alpha = annualize_daily_intercept(terminal_intercept)
    terminal_regression_anns[regime_name] = ann_alpha
    subplot_titles.append(f"{regime_name} Regime · Market vs Secret Strategy")
    # ---- CHANGED: Remove & before alpha here ----
    subtitle = (
        f"{regime_name} Regime · Strategy Return vs Market Return<br>"
        f"<span style='font-size:12px'>α<sub>ann</sub> = {ann_alpha:.1%}</span>"
    )
    subplot_titles.append(subtitle)

fig = make_subplots(
    rows=3,
    cols=2,
    row_heights=[0.333, 0.333, 0.333],
    column_widths=[0.54, 0.46],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
    subplot_titles=tuple(subplot_titles),
    specs=[[{}, {}], [{}, {}], [{}, {}]],
)

trace_indices = {}

for row, regime_name in enumerate(regime_names, start=1):
    d = regime_dfs[regime_name]
    initial_slice = d.iloc[: initial_i + 1]
    initial_returns = slice_returns(d, initial_i)

    beta, intercept, corr, r2 = regression_stats(
        initial_returns["Market Return"],
        initial_returns["Strategy Return"],
    )
    reg_y = intercept + beta * reg_x

    # Left chart: indexed performance in this regime.
    market_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=initial_slice["date"],
            y=initial_slice["Market"],
            mode="lines",
            line=dict(color=market_color, width=3),
            name="Market" if row == 1 else "Market",
            showlegend=(row == 1),
            legendgroup="market",
            hovertemplate="Date: %{x|%Y-%m-%d}<br>Market: %{y:.2f}<extra></extra>",
        ),
        row=row,
        col=1,
    )

    glow_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=initial_slice["date"],
            y=initial_slice["Secret Strategy"],
            mode="lines",
            line=dict(color=strategy_glow_color, width=12),
            name="Secret Strategy Glow",
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=1,
    )

    strategy_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=initial_slice["date"],
            y=initial_slice["Secret Strategy"],
            mode="lines",
            line=dict(color=strategy_color, width=4),
            name="Secret Strategy" if row == 1 else "Secret Strategy",
            showlegend=(row == 1),
            legendgroup="strategy",
            hovertemplate="Date: %{x|%Y-%m-%d}<br>Secret Strategy: %{y:.2f}<extra></extra>",
        ),
        row=row,
        col=1,
    )

    # Right chart: return scatter and OLS line for this regime only.
    scatter_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=initial_returns["Market Return"],
            y=initial_returns["Strategy Return"],
            mode="markers",
            marker=dict(size=6, color=scatter_color, line=dict(width=0)),
            customdata=initial_returns["date"],
            name=f"{regime_name} Return Scatter",
            showlegend=False,
            hovertemplate=(
                "Date: %{customdata|%Y-%m-%d}<br>"
                "Market Return: %{x:.2%}<br>"
                "Secret Strategy Return: %{y:.2%}<extra></extra>"
            ),
        ),
        row=row,
        col=2,
    )

    reg_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=reg_x,
            y=reg_y,
            mode="lines",
            line=dict(color=regression_color, width=3, dash="dash"),
            name=f"{regime_name} OLS Fit",
            showlegend=False,
            hovertemplate="OLS fit<extra></extra>",
        ),
        row=row,
        col=2,
    )

    # Remove the overlay text trace (regression metrics)

    trace_indices[regime_name] = {
        "market": market_trace_idx,
        "glow": glow_trace_idx,
        "strategy": strategy_trace_idx,
        "scatter": scatter_trace_idx,
        "regression": reg_trace_idx,
        # "text": text_trace_idx, # text trace is removed
    }

    fig.add_hline(
        y=0,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=row,
        col=2,
    )
    fig.add_vline(
        x=0,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=row,
        col=2,
    )

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []
trace_order = []
for regime_name in regime_names:
    trace_order.extend(
        [
            trace_indices[regime_name]["market"],
            trace_indices[regime_name]["glow"],
            trace_indices[regime_name]["strategy"],
            trace_indices[regime_name]["scatter"],
            trace_indices[regime_name]["regression"],
            # trace_indices[regime_name]["text"], # text traces removed
        ]
    )

for i in frame_indices:
    frame_name = f"f{i}"
    frame_data = []

    for regime_name in regime_names:
        d = regime_dfs[regime_name]
        current_slice = d.iloc[: i + 1]
        current_returns = slice_returns(d, i)

        beta, intercept, corr, r2 = regression_stats(
            current_returns["Market Return"],
            current_returns["Strategy Return"],
        )
        reg_y = intercept + beta * reg_x

        frame_data.extend(
            [
                go.Scatter(
                    x=current_slice["date"],
                    y=current_slice["Market"],
                    line=dict(color=market_color, width=3),
                ),
                go.Scatter(
                    x=current_slice["date"],
                    y=current_slice["Secret Strategy"],
                    line=dict(color=strategy_glow_color, width=12),
                ),
                go.Scatter(
                    x=current_slice["date"],
                    y=current_slice["Secret Strategy"],
                    line=dict(color=strategy_color, width=4),
                ),
                go.Scatter(
                    x=current_returns["Market Return"],
                    y=current_returns["Strategy Return"],
                    marker=dict(size=6, color=scatter_color, line=dict(width=0)),
                    customdata=current_returns["date"],
                ),
                go.Scatter(x=reg_x, y=reg_y),
                # Remove text overlay from animation frame
            ]
        )

    frames.append(
        go.Frame(
            data=frame_data,
            traces=trace_order,
            name=frame_name,
        )
    )

    elapsed_years = i / TRADING_DAYS_PER_YEAR
    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                    "fromcurrent": True,
                },
            ],
            "label": f"{elapsed_years:.1f}Y",
            "method": "animate",
        }
    )

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Secret Strategy Relative to Market Across Regimes"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=900,
    width=1200,
    margin=dict(t=130, b=135, r=55, l=75),
    legend=dict(
        orientation="v",
        x=0,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": FRAME_DURATION, "redraw": False},
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {
                "font": {"size": 14, "color": off_white},
                "prefix": "Through: ",
                "visible": True,
                "xanchor": "right",
            },
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 50},
            "len": 0.85,
            "x": 0.15,
            "y": 0,
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=13))

# Axes
for row, regime_name in enumerate(regime_names, start=1):
    d = regime_dfs[regime_name]

    fig.update_xaxes(
        axis_style,
        row=row,
        col=1,
        range=[d["date"].iloc[0], d["date"].iloc[-1]],
        title_text="Date" if row == 3 else "",
    )
    fig.update_yaxes(
        axis_style,
        row=row,
        col=1,
        range=price_ranges[regime_name],
        title_text="Index Value",
    )

    fig.update_xaxes(
        axis_style,
        row=row,
        col=2,
        range=scatter_x_range,
        title_text="Market Daily Return" if row == 3 else "",
        tickformat=".1%",
    )
    fig.update_yaxes(
        axis_style,
        row=row,
        col=2,
        range=scatter_y_range,
        title_text="Strategy Daily Return",
        tickformat=".1%",
    )

# ============================================================
# Save / show
# ============================================================

fig.show()


---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
  - This notebook explores why the secrecy of trading strategies is often overrated and why most successful approaches rely on systematic risk exposure (“risk premia”) rather than hidden inefficiencies or secret “alpha.”
  - We examine how return streams in financial markets are largely driven by broad, undiversifiable risk factors—such as equity, volatility, value, momentum, credit, and liquidity—and how these are widely known and accessible to most participants.
  - Factor models and the academic literature show that idiosyncratic (truly “secret”) alpha is rare, fleeting, and difficult to systematically capture; meaningful returns typically come from thoughtful allocation and skilled risk-taking.
  - The analogy of allocating risk to “stepping up to the plate” emphasizes that, while we can’t control market regimes (the “pitcher”), experience and positioning allow us to harvest rewards from the available risk premia.
  - Key takeaway: Durable trading success comes not from always outsmarting the crowd or hiding edge, but from intelligent risk allocation within the landscape of known return drivers—meaning, strategies don’t need to be secretive to be effective.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$